In [4]:
from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters
from lignin_saf.ligsaf_system import create_rcf_system
from lignin_saf.ligsaf_purification_system import create_rcf_oil_purification_system
from lignin_saf.monomer_purification import create_monomer_purification_system
from lignin_saf.ligsaf_utilities_system import create_rcf_utilities_system
from cellulosic_tea import create_cellulosic_ethanol_tea
from biosteam import main_flowsheet as F
import biosteam as bst
import thermosteam as tmo


In [5]:
# Code just to increase the number of display units for the various components
tmo.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
tmo.MultiStream.display_units.N = 40  
bst.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
bst.MultiStream.display_units.N = 40  

In [6]:
chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7   # 2016 USD basis

# Poplar group must be defined before creating any stream that references it
chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d')



c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: Poplar_In> has been replaced in registry
  self._register(ID)


In [7]:
rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()

rcf_oil_purification_sys = create_rcf_oil_purification_system(ins=F.RCF_Oil)
monomer_purification_sys = create_monomer_purification_system(ins=F.Purified_RCF_Oil)
rcf_oil_purification_sys.simulate()
monomer_purification_sys.simulate()
BT, WWT, gas_mixer = create_rcf_utilities_system()



rcf_combined_system = bst.System(
    'Combined_RCF_System',
    path=(rcf_system, rcf_oil_purification_sys, monomer_purification_sys, WWT),
    facilities=[gas_mixer, BT],
)
rcf_combined_system.simulate()
rcf_combined_system.show()



#integrated_tea = create_cellulosic_ethanol_tea(rcf_combined_system)

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_multi_stream.py:251: RuntimeWarning: <Stream: Meoh_recycle> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <MultiStream: hydrogen_recycle> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: Meoh_in> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: Hydrogen_In> has been replaced in registry
  self._register(ID)
C:\Users\hwadg\AppData\Local\Temp\ipykernel_10248\3256742016.py:1: RuntimeWarning: <Mixer: MIX100> has been replaced in registry
  rcf_system = create_rcf_system(ins=poplar_in)
C:\Users\hwadg\AppData\Local\Temp\ipykernel_10248\3256742016.py:1: RuntimeWarning: <Pump: PUMP101> has been replaced in re

System: Combined_RCF_System
Highest convergence error among components in recycle
streams {HX117-0, PUMP112-0} after 1 loops:
- flow rate   1.57e-08 kmol/hr (0.017%)
- temperature 2.35e-03 K (0.00076%)
ins...
[0] RCF_Catalyst  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): NiC  2.28
[1] air_lagoon  
    phase: 'g', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): N2  281
                    O2  69.4
[2] caustic  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water  58.3
                    NaOH   26.3
[3] Poplar_In  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water     925
                    Sucrose   0.243
                    Extract   7.4
                    Acetate   48.6
                    Ash       1e+03
                    Lignin    156
                    Glucan    238
                    Xylan     84.5
                    Arabinan  1.26
                    Mannan    19
                    Galactan  7.2
[4] Meoh_in  
    phase: '

In [8]:
BT

BoilerTurbogenerator: BT
ins...
[0] sludge  from  SludgeCentrifuge-S603
    phase: 'l', T: 307.97 K, P: 101325 Pa
    flow (kmol/hr): Water           1.3e+03
                    Extract         0.00313
                    Acetate         1.19
                    NaOH            0.4
                    Glucan          18.1
                    Xylan           4.53
                    Arabinan        0.726
                    Mannan          9.12
                    Galactan        3.45
                    WWTsludge       20.7
                    Methanol        0.00426
                    Hexane          0.00045
                    EthylAcetate    0.0055
                    Propylguaiacol  3.79e-06
                    Propylsyringol  3.21e-06
                    Syringaresinol  7.53e-07
                    G_Dimer         8.71e-07
                    S_Oligomer      0.00158
                    G_Oligomer      0.00179
[1] s86  from  Mixer-MIX_BT_gas
    phase: 'g', T: 396.39 K, P: 101325 

In [9]:
WWT

System: WWT
Highest convergence error among components in recycle
stream M604-0 after 5 loops:
- flow rate   3.70e-01 kmol/hr (0.062%)
- temperature 7.96e-05 K (2.6e-05%)
ins...
[0] WW_10  from  MultiStageMixerSettlers-LLE200
    phase: 'l', T: 318.72 K, P: 101325 Pa
    flow (kmol/hr): Water           1.52e+03
                    Extract         7.4
                    Glucan          23.8
                    Xylan           5.92
                    Arabinan        0.757
                    Mannan          9.51
                    Galactan        3.6
                    Propylguaiacol  1.59e-06
                    Propylsyringol  1.35e-06
                    Syringaresinol  2.08e-08
                    G_Dimer         2.17e-06
                    S_Oligomer      2.16e-07
                    G_Oligomer      2.45e-07
[1] WastePulp  from  LiquidsSplitCentrifuge-CENT203
    phase: 'l', T: 350.22 K, P: 101325 Pa
    flow (kmol/hr): Water         0.00274
                    EthylAcetate  15